In [50]:
import pandas as pd
import numpy as np

In [51]:
plant1 = pd.read_csv("../data/processed/plant1_merged.csv")
plant2 = pd.read_csv("../data/processed/plant2_merged.csv")

# Convert DATE_TIME back to datetime
plant1["DATE_TIME"] = pd.to_datetime(plant1["DATE_TIME"])
plant2["DATE_TIME"] = pd.to_datetime(plant2["DATE_TIME"])

print("Plant 1 shape:", plant1.shape)
print("Plant 2 shape:", plant2.shape)

Plant 1 shape: (68774, 10)
Plant 2 shape: (67698, 10)


In [52]:
# Function to create time-based features
def create_time_features(df):
    df = df.copy()

    df["hour"] = df["DATE_TIME"].dt.hour
    df["minute"] = df["DATE_TIME"].dt.minute
    df["day"] = df["DATE_TIME"].dt.day
    df["day_of_week"] = df["DATE_TIME"].dt.dayofweek
    df["day_of_year"] = df["DATE_TIME"].dt.dayofyear
    df["month"] = df["DATE_TIME"].dt.month

    return df


# Apply to both plants
plant1 = create_time_features(plant1)
plant2 = create_time_features(plant2)

print("Plant 1:")
print(plant1[[
    "DATE_TIME",
    "hour",
    "minute",
    "day",
    "day_of_week",
    "day_of_year",
    "month"
]].head())

print("\nPlant 2:")
print(plant2[[
    "DATE_TIME",
    "hour",
    "minute",
    "day",
    "day_of_week",
    "day_of_year",
    "month"
]].head())

Plant 1:
   DATE_TIME  hour  minute  day  day_of_week  day_of_year  month
0 2020-05-15     0       0   15            4          136      5
1 2020-05-15     0       0   15            4          136      5
2 2020-05-15     0       0   15            4          136      5
3 2020-05-15     0       0   15            4          136      5
4 2020-05-15     0       0   15            4          136      5

Plant 2:
   DATE_TIME  hour  minute  day  day_of_week  day_of_year  month
0 2020-05-15     0       0   15            4          136      5
1 2020-05-15     0       0   15            4          136      5
2 2020-05-15     0       0   15            4          136      5
3 2020-05-15     0       0   15            4          136      5
4 2020-05-15     0       0   15            4          136      5


In [53]:
# Cyclic encoding for time of day
plant1["hour_sin"] = np.sin(
    2 * np.pi * (plant1["hour"] + plant1["minute"] / 60) / 24
)

plant1["hour_cos"] = np.cos(
    2 * np.pi * (plant1["hour"] + plant1["minute"] / 60) / 24
)

plant2["hour_sin"] = np.sin(
    2 * np.pi * (plant2["hour"] + plant2["minute"] / 60) / 24
)

plant2["hour_cos"] = np.cos(
    2 * np.pi * (plant2["hour"] + plant2["minute"] / 60) / 24
)

print(plant1[[
    "DATE_TIME",
    "hour",
    "minute",
    "hour_sin",
    "hour_cos"
]].head(10))

   DATE_TIME  hour  minute  hour_sin  hour_cos
0 2020-05-15     0       0       0.0       1.0
1 2020-05-15     0       0       0.0       1.0
2 2020-05-15     0       0       0.0       1.0
3 2020-05-15     0       0       0.0       1.0
4 2020-05-15     0       0       0.0       1.0
5 2020-05-15     0       0       0.0       1.0
6 2020-05-15     0       0       0.0       1.0
7 2020-05-15     0       0       0.0       1.0
8 2020-05-15     0       0       0.0       1.0
9 2020-05-15     0       0       0.0       1.0


In [54]:
# Sort data by inverter and timestamp
plant1 = plant1.sort_values(
    ["SOURCE_KEY", "DATE_TIME"]
).reset_index(drop=True)

plant2 = plant2.sort_values(
    ["SOURCE_KEY", "DATE_TIME"]
).reset_index(drop=True)


# Create lag features separately for each inverter
for lag in [1, 4, 8, 96]:
    plant1[f"AC_POWER_lag_{lag}"] = (
        plant1.groupby("SOURCE_KEY")["AC_POWER"].shift(lag)
    )

    plant2[f"AC_POWER_lag_{lag}"] = (
        plant2.groupby("SOURCE_KEY")["AC_POWER"].shift(lag)
    )


print("Plant 1 lag features created:")
print(
    plant1[
        [
            "SOURCE_KEY",
            "DATE_TIME",
            "AC_POWER",
            "AC_POWER_lag_1",
            "AC_POWER_lag_4",
            "AC_POWER_lag_8",
            "AC_POWER_lag_96"
        ]
    ].head(110)
)

Plant 1 lag features created:
          SOURCE_KEY           DATE_TIME  AC_POWER  AC_POWER_lag_1  \
0    1BY6WEcLGh8j5v7 2020-05-15 00:00:00  0.000000             NaN   
1    1BY6WEcLGh8j5v7 2020-05-15 00:15:00  0.000000             0.0   
2    1BY6WEcLGh8j5v7 2020-05-15 00:30:00  0.000000             0.0   
3    1BY6WEcLGh8j5v7 2020-05-15 00:45:00  0.000000             0.0   
4    1BY6WEcLGh8j5v7 2020-05-15 01:00:00  0.000000             0.0   
..               ...                 ...       ...             ...   
105  1BY6WEcLGh8j5v7 2020-05-16 05:00:00  0.000000             0.0   
106  1BY6WEcLGh8j5v7 2020-05-16 05:15:00  0.000000             0.0   
107  1BY6WEcLGh8j5v7 2020-05-16 05:30:00  0.000000             0.0   
108  1BY6WEcLGh8j5v7 2020-05-16 05:45:00  0.000000             0.0   
109  1BY6WEcLGh8j5v7 2020-05-16 06:00:00  4.757143             0.0   

     AC_POWER_lag_4  AC_POWER_lag_8  AC_POWER_lag_96  
0               NaN             NaN              NaN  
1               NaN

In [55]:
# Create rolling features separately for each inverter
plant1["AC_POWER_rolling_mean_4"] = (
    plant1.groupby("SOURCE_KEY")["AC_POWER"]
    .transform(lambda x: x.shift(1).rolling(window=4).mean())
)

plant1["AC_POWER_rolling_mean_8"] = (
    plant1.groupby("SOURCE_KEY")["AC_POWER"]
    .transform(lambda x: x.shift(1).rolling(window=8).mean())
)


plant2["AC_POWER_rolling_mean_4"] = (
    plant2.groupby("SOURCE_KEY")["AC_POWER"]
    .transform(lambda x: x.shift(1).rolling(window=4).mean())
)

plant2["AC_POWER_rolling_mean_8"] = (
    plant2.groupby("SOURCE_KEY")["AC_POWER"]
    .transform(lambda x: x.shift(1).rolling(window=8).mean())
)


print(
    plant1[
        [
            "SOURCE_KEY",
            "DATE_TIME",
            "AC_POWER",
            "AC_POWER_rolling_mean_4",
            "AC_POWER_rolling_mean_8"
        ]
    ].head(12)
)

         SOURCE_KEY           DATE_TIME  AC_POWER  AC_POWER_rolling_mean_4  \
0   1BY6WEcLGh8j5v7 2020-05-15 00:00:00       0.0                      NaN   
1   1BY6WEcLGh8j5v7 2020-05-15 00:15:00       0.0                      NaN   
2   1BY6WEcLGh8j5v7 2020-05-15 00:30:00       0.0                      NaN   
3   1BY6WEcLGh8j5v7 2020-05-15 00:45:00       0.0                      NaN   
4   1BY6WEcLGh8j5v7 2020-05-15 01:00:00       0.0                      0.0   
5   1BY6WEcLGh8j5v7 2020-05-15 01:15:00       0.0                      0.0   
6   1BY6WEcLGh8j5v7 2020-05-15 01:30:00       0.0                      0.0   
7   1BY6WEcLGh8j5v7 2020-05-15 01:45:00       0.0                      0.0   
8   1BY6WEcLGh8j5v7 2020-05-15 02:00:00       0.0                      0.0   
9   1BY6WEcLGh8j5v7 2020-05-15 02:15:00       0.0                      0.0   
10  1BY6WEcLGh8j5v7 2020-05-15 02:30:00       0.0                      0.0   
11  1BY6WEcLGh8j5v7 2020-05-15 02:45:00       0.0               

In [56]:
# Create 15-minute-ahead forecasting target
plant1["TARGET_AC_POWER"] = (
    plant1.groupby("SOURCE_KEY")["AC_POWER"].shift(-1)
)

plant2["TARGET_AC_POWER"] = (
    plant2.groupby("SOURCE_KEY")["AC_POWER"].shift(-1)
)


print(
    plant1[
        [
            "SOURCE_KEY",
            "DATE_TIME",
            "AC_POWER",
            "TARGET_AC_POWER"
        ]
    ].head(10)
)

        SOURCE_KEY           DATE_TIME  AC_POWER  TARGET_AC_POWER
0  1BY6WEcLGh8j5v7 2020-05-15 00:00:00       0.0              0.0
1  1BY6WEcLGh8j5v7 2020-05-15 00:15:00       0.0              0.0
2  1BY6WEcLGh8j5v7 2020-05-15 00:30:00       0.0              0.0
3  1BY6WEcLGh8j5v7 2020-05-15 00:45:00       0.0              0.0
4  1BY6WEcLGh8j5v7 2020-05-15 01:00:00       0.0              0.0
5  1BY6WEcLGh8j5v7 2020-05-15 01:15:00       0.0              0.0
6  1BY6WEcLGh8j5v7 2020-05-15 01:30:00       0.0              0.0
7  1BY6WEcLGh8j5v7 2020-05-15 01:45:00       0.0              0.0
8  1BY6WEcLGh8j5v7 2020-05-15 02:00:00       0.0              0.0
9  1BY6WEcLGh8j5v7 2020-05-15 02:15:00       0.0              0.0


In [57]:
feature_columns = [
    "AC_POWER_lag_1",
    "AC_POWER_lag_4",
    "AC_POWER_lag_8",
    "AC_POWER_lag_96",
    "AC_POWER_rolling_mean_4",
    "AC_POWER_rolling_mean_8",
    "TARGET_AC_POWER"
]

print("Plant 1 missing values:")
print(plant1[feature_columns].isnull().sum())

print("\nPlant 2 missing values:")
print(plant2[feature_columns].isnull().sum())

Plant 1 missing values:
AC_POWER_lag_1               22
AC_POWER_lag_4               88
AC_POWER_lag_8              176
AC_POWER_lag_96            2112
AC_POWER_rolling_mean_4      88
AC_POWER_rolling_mean_8     176
TARGET_AC_POWER              22
dtype: int64

Plant 2 missing values:
AC_POWER_lag_1               22
AC_POWER_lag_4               88
AC_POWER_lag_8              176
AC_POWER_lag_96            2112
AC_POWER_rolling_mean_4      88
AC_POWER_rolling_mean_8     176
TARGET_AC_POWER              22
dtype: int64


In [58]:
# Remove rows where required features or target are unavailable

required_columns = feature_columns

plant1 = plant1.dropna(subset=required_columns).reset_index(drop=True)
plant2 = plant2.dropna(subset=required_columns).reset_index(drop=True)

print("Plant 1 shape after removing missing values:", plant1.shape)
print("Plant 2 shape after removing missing values:", plant2.shape)

Plant 1 shape after removing missing values: (66640, 25)
Plant 2 shape after removing missing values: (65564, 25)


In [59]:
# Final missing-value validation

print("Plant 1 missing values:")
print(plant1[required_columns].isnull().sum())

print("\nPlant 2 missing values:")
print(plant2[required_columns].isnull().sum())

Plant 1 missing values:
AC_POWER_lag_1             0
AC_POWER_lag_4             0
AC_POWER_lag_8             0
AC_POWER_lag_96            0
AC_POWER_rolling_mean_4    0
AC_POWER_rolling_mean_8    0
TARGET_AC_POWER            0
dtype: int64

Plant 2 missing values:
AC_POWER_lag_1             0
AC_POWER_lag_4             0
AC_POWER_lag_8             0
AC_POWER_lag_96            0
AC_POWER_rolling_mean_4    0
AC_POWER_rolling_mean_8    0
TARGET_AC_POWER            0
dtype: int64


In [60]:
# Define final features and target

feature_columns = [
    "hour",
    "minute",
    "day",
    "day_of_week",
    "day_of_year",
    "month",
    "hour_sin",
    "hour_cos",
    "AC_POWER_lag_1",
    "AC_POWER_lag_4",
    "AC_POWER_lag_8",
    "AC_POWER_lag_96",
    "AC_POWER_rolling_mean_4",
    "AC_POWER_rolling_mean_8"
]

target_column = "TARGET_AC_POWER"

print("Number of features:", len(feature_columns))
print("\nFeature columns:")
print(feature_columns)

print("\nTarget column:")
print(target_column)

Number of features: 14

Feature columns:
['hour', 'minute', 'day', 'day_of_week', 'day_of_year', 'month', 'hour_sin', 'hour_cos', 'AC_POWER_lag_1', 'AC_POWER_lag_4', 'AC_POWER_lag_8', 'AC_POWER_lag_96', 'AC_POWER_rolling_mean_4', 'AC_POWER_rolling_mean_8']

Target column:
TARGET_AC_POWER


In [61]:
# Final X and y validation

X1 = plant1[feature_columns]
y1 = plant1[target_column]

X2 = plant2[feature_columns]
y2 = plant2[target_column]

print("Plant 1:")
print("X shape:", X1.shape)
print("y shape:", y1.shape)

print("\nPlant 2:")
print("X shape:", X2.shape)
print("y shape:", y2.shape)

print("\nPlant 1 X missing:", X1.isnull().sum().sum())
print("Plant 1 y missing:", y1.isnull().sum())

print("\nPlant 2 X missing:", X2.isnull().sum().sum())
print("Plant 2 y missing:", y2.isnull().sum())

Plant 1:
X shape: (66640, 14)
y shape: (66640,)

Plant 2:
X shape: (65564, 14)
y shape: (65564,)

Plant 1 X missing: 0
Plant 1 y missing: 0

Plant 2 X missing: 0
Plant 2 y missing: 0


In [62]:
# Verify chronological ordering within each inverter

print("Plant 1 chronological order:",
      plant1.groupby("SOURCE_KEY")["DATE_TIME"].apply(lambda x: x.is_monotonic_increasing).all())

print("Plant 2 chronological order:",
      plant2.groupby("SOURCE_KEY")["DATE_TIME"].apply(lambda x: x.is_monotonic_increasing).all())

Plant 1 chronological order: True
Plant 2 chronological order: True


In [63]:
import os

FINAL_PATH = "../data/final/"
os.makedirs(FINAL_PATH, exist_ok=True)

In [64]:
# Save final engineered datasets

plant1.to_csv(
    FINAL_PATH + "plant1_features.csv",
    index=False
)

plant2.to_csv(
    FINAL_PATH + "plant2_features.csv",
    index=False
)

print("Engineered datasets saved successfully.")

Engineered datasets saved successfully.


In [65]:
# Verify saved files

print("Plant 1 saved:", os.path.exists(FINAL_PATH + "plant1_features.csv"))
print("Plant 2 saved:", os.path.exists(FINAL_PATH + "plant2_features.csv"))

Plant 1 saved: True
Plant 2 saved: True
